# 013 Serving and Adapters

这是 LangGraph 学习线的第十三份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langgraph/streaming
- https://docs.langchain.com/oss/python/langgraph/interrupts
- https://fastapi.tiangolo.com/advanced/custom-response/#streamingresponse

学习目标：

1. 理解 LangGraph 图本身不是 HTTP 接口，`CompiledStateGraph` 需要被 Web 层适配后才能给前端或业务系统调用
2. 学会把 FastAPI 请求转换成 LangGraph 的 `input`、`config`、`context`
3. 理解 `session_id` 和 `thread_id` 的映射关系，以及它们如何影响 checkpointer 读取历史状态
4. 学会消费 `agent.astream(...)` 的 `messages`、`updates`、`custom` 等 stream mode
5. 学会把 LangGraph 内部事件转换成前端 SSE 业务事件
6. 理解 `Command(resume=...)` 如何把 HTTP 请求映射成 LangGraph interrupt 恢复
7. 对比本仓库 `hb_rs/endpoint.py`：为什么它不是图定义文件，而是 LangGraph Web 适配层

这一讲不启动真实 FastAPI 服务，而是用普通函数模拟一个 Web 适配器。重点是看清边界。

## 1. 为什么需要适配器

前面几讲主要学习的是：

```text
StateGraph -> compile -> invoke / stream
```

但真实后端系统面对的是：

```text
HTTP 请求 -> 鉴权/参数校验 -> 会话 ID -> 流式响应 -> 前端业务事件
```

LangGraph 不直接规定你的 HTTP 协议。它只提供图运行能力。

所以中间需要一个适配层：

```text
FastAPI Request
  -> Adapter
  -> LangGraph input/config/context
  -> graph.stream(...) / graph.astream(...)
  -> Adapter
  -> SSE event
```

Java 类比：

```text
LangGraph compiled graph 像一个业务 Service。
endpoint.py 像 Controller + DTO Mapper + SSEEmitter Adapter。
```

适配器的价值不是让 agent 更聪明，而是让 agent 能稳定接入现有 Web 系统。

In [ ]:
import importlib.metadata
import json
from dataclasses import dataclass
from operator import add
from typing import Annotated, Any

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from typing_extensions import TypedDict

print("langgraph", importlib.metadata.version("langgraph"))

## 2. 先定义一个最小客服图

这个图只有一个节点：

```text
START -> answer -> END
```

它模拟客服回答，并通过 `get_stream_writer()` 发一个 `custom` 事件。

`messages` 使用 reducer：

```python
Annotated[list[dict], add]
```

表示同一个 `thread_id` 下，新消息会追加到历史消息后面。

In [ ]:
class ServiceState(TypedDict):
    messages: Annotated[list[dict], add]
    final_answer: str


def answer_node(state: ServiceState) -> dict:
    writer = get_stream_writer()
    writer({"type": "progress", "stage": "answer", "message": "开始生成回答"})

    user_messages = [
        message["content"]
        for message in state.get("messages", [])
        if message.get("role") == "user"
    ]
    latest_question = user_messages[-1] if user_messages else ""
    answer = "我收到了你的问题：" + latest_question

    return {
        "final_answer": answer,
        "messages": [{"role": "assistant", "content": answer}],
    }


checkpointer = InMemorySaver()

service_graph = (
    StateGraph(ServiceState)
    .add_node("answer", answer_node)
    .add_edge(START, "answer")
    .add_edge("answer", END)
    .compile(checkpointer=checkpointer)
)

## 3. HTTP 请求不能直接丢给 graph

Web 请求通常长这样：

```json
{
  "query": "你好",
  "session_id": "user-session-001",
  "stream": true
}
```

LangGraph 需要的是：

```python
input = {"messages": [{"role": "user", "content": query}]}
config = {"configurable": {"thread_id": session_id}}
```

这一步就是 Request Mapper。

In [ ]:
@dataclass
class ChatRequest:
    query: str | None = None
    resume: dict | None = None
    session_id: str = "default-session"
    stream: bool = True


def to_langgraph_call(request: ChatRequest) -> tuple[Any, dict]:
    if request.query and request.resume:
        raise ValueError("query 和 resume 不能同时存在")

    config = {"configurable": {"thread_id": request.session_id}}

    if request.query:
        graph_input = {
            "messages": [
                {"role": "user", "content": request.query},
            ]
        }
    elif request.resume:
        graph_input = Command(resume=request.resume)
    else:
        graph_input = None

    return graph_input, config

## 4. `session_id` 为什么要映射成 `thread_id`

LangGraph 的 checkpointer 按 `thread_id` 保存状态。

业务接口一般叫：

```text
session_id / conversation_id / chat_id
```

LangGraph 叫：

```text
thread_id
```

适配器要把二者对齐：

```text
request.session_id -> config.configurable.thread_id
```

否则第二轮请求就找不到第一轮保存的 graph state。

In [ ]:
turn_1 = ChatRequest(query="我的名字是 Bob", session_id="serving-demo")
input_1, config_1 = to_langgraph_call(turn_1)
result_1 = service_graph.invoke(input_1, config_1)

turn_2 = ChatRequest(query="我刚才说了什么？", session_id="serving-demo")
input_2, config_2 = to_langgraph_call(turn_2)
result_2 = service_graph.invoke(input_2, config_2)

print("message count:", len(result_2["messages"]))
for message in result_2["messages"]:
    print(message["role"], "=>", message["content"])

## 5. Graph stream 也不能直接给前端

`graph.stream(...)` / `graph.astream(...)` 输出的是 LangGraph 运行时事件。

前端通常需要的是业务事件：

```text
token
tool_calls
tool_output
quick_entries
suggested_questions
```

所以适配器还要做 Stream Event Mapping：

```text
LangGraph part -> StreamResponse -> SSE text
```

下面用 `updates` 和 `custom` 两种事件演示。

In [ ]:
def encode_sse(event: str, data: Any) -> str:
    payload = {"event": event, "data": data}
    return "data: " + json.dumps(payload, ensure_ascii=False) + "\n\n"


def map_stream_part_to_sse(part: dict) -> str:
    if part["type"] == "custom":
        return encode_sse("custom", part["data"])

    if part["type"] == "updates":
        return encode_sse("graph_update", part["data"])

    return encode_sse("other", part)

## 6. 模拟一次 SSE 流式输出

真实 FastAPI 里会写：

```python
return StreamingResponse(event_stream(), media_type="text/event-stream")
```

Notebook 里不启动服务，只收集并打印适配后的 SSE 文本。

In [ ]:
def collect_sse_events() -> list[str]:
    request = ChatRequest(query="帮我解释 LangGraph 适配器", session_id="stream-demo")
    graph_input, config = to_langgraph_call(request)

    events = []
    for part in service_graph.stream(
        graph_input,
        config,
        stream_mode=["updates", "custom"],
        version="v2",
    ):
        events.append(map_stream_part_to_sse(part))
    return events


for event in collect_sse_events():
    print(event)

## 7. `Command(resume=...)` 也是适配器职责

如果 graph 里使用了 `interrupt()`，第一次执行会暂停。

用户后续通过 HTTP 提交审批结果时，请求一般长这样：

```json
{
  "session_id": "approval-session-001",
  "resume": {"approved": true}
}
```

适配器要把它转换成：

```python
Command(resume={"approved": True})
```

同时继续使用同一个 `thread_id`。

这就是为什么通用 endpoint 里会同时支持 `query` 和 `resume`，并且禁止二者同时存在。

In [ ]:
resume_request = ChatRequest(
    resume={"approved": True},
    session_id="approval-session-001",
)

resume_input, resume_config = to_langgraph_call(resume_request)

print(type(resume_input).__name__)
print(resume_config)

## 8. 一个 FastAPI 适配器的形状

真实代码大概会长这样：

```python
@app.post("/chat")
async def chat(request: ChatRequest):
    graph_input, config = to_langgraph_call(request)

    async def event_stream():
        async for part in graph.astream(
            graph_input,
            config,
            stream_mode=["messages", "updates", "custom"],
        ):
            yield map_stream_part_to_sse(part)

    return StreamingResponse(event_stream(), media_type="text/event-stream")
```

注意这里还是没有定义图。

它只是把一个已经编译好的 graph 暴露成 HTTP SSE 接口。

## 9. 和 `hb_rs/endpoint.py` 的对应关系

| 本讲概念 | `hb_rs/endpoint.py` 中的实现 |
| --- | --- |
| 请求 DTO | `GeneralAPIRequest` |
| 响应 DTO | `StreamResponse` |
| 注册接口 | `add_general_api_endpoint(...)` |
| 业务 context 拼接 | `class Request(GeneralAPIRequest, context)` |
| `session_id -> thread_id` | `config = {"configurable": {"thread_id": request.session_id}}` |
| 普通提问 input | `{"messages": [{"role": "user", "content": request.query}]}` |
| interrupt 恢复 input | `Command(resume=request.resume)` |
| graph 执行 | `agent.astream(...)` |
| LangGraph 事件转 SSE | `mode == "messages" / "updates" / "custom"` 分支 |
| 自定义业务事件 | `quick_entries` |
| 可观测性 | `langfuse_request_scope(...)` 和 callback config |

所以 `endpoint.py` 不属于图编排代码。

它属于：

```text
LangGraph Web Adapter
```

## 10. 为什么不用框架完全替代

LangServe、LangGraph Platform、LangGraph SDK 都能提供一部分 serving 能力。

但业务系统通常还有自己的协议：

- 现有 URL，例如 `/admin_api/customer/customer`
- 现有登录态、网关、权限
- 前端约定好的 SSE event 结构
- 自定义事件，例如 `quick_entries`、`suggested_questions`
- 业务日志和可观测性字段
- 兼容已有请求参数

所以真实项目经常会保留自己的薄适配器。

合理目标不是消灭适配器，而是让适配器职责清楚：

```text
Controller 只管 HTTP
RequestMapper 只管请求到 graph input
StreamMapper 只管 graph event 到 SSE event
Observability 只管 trace/span
业务 middleware 只管 agent 行为
```

## 11. 本讲练习

请判断下面逻辑应该放在哪里：

1. 把 `session_id` 放进 `config.configurable.thread_id`
2. 根据用户问题做 RAG 检索
3. 把 `mode == "custom"` 且 `type == "quick_entries"` 转成 SSE `quick_entries`
4. 判断模型是否需要调用工具
5. 把 Langfuse trace id 写进日志
6. 把 `resume` 请求转换成 `Command(resume=...)`

参考答案：

1. RequestMapper / endpoint adapter
2. RAG middleware / graph node
3. StreamMapper / endpoint adapter
4. Agent model/tool loop 或业务 middleware
5. Observability wrapper
6. RequestMapper / endpoint adapter

## 12. 本讲小结

这一讲的核心：

```text
LangGraph 负责运行图。
Adapter 负责把图接入 Web 系统。
```

你现在应该能看懂：

- 为什么 `CompiledStateGraph` 不是 HTTP 接口
- 为什么要把 `session_id` 映射成 `thread_id`
- 为什么 `query` 要转成 `messages`
- 为什么 `resume` 要转成 `Command(resume=...)`
- 为什么 `agent.astream(...)` 的结果还要再映射成 SSE
- 为什么 `endpoint.py` 不是图定义文件，而是 Web 适配层

下一步再看真实项目时，先问三个问题：

```text
图在哪里定义？
图在哪里执行？
图的事件在哪里被转换成前端协议？
```